# Tutorial 1: the Metabolic Dataset and Analysis Workspace

This notebook introduces the inputs that form a Metabis **Metabolic Dataset**:

- `MetabolicModel`: the packaged reference model and its identifiers
- `ReactionData`: the reaction activity matrix aligned to the model
- `ExpressionData`: the gene expression matrix translated to model identifiers
- `SampleMetadata`: independently loaded sample annotations aligned by canonical sample ID

It then combines those inputs in an `AnalysisWorkspace`, the object that coordinates revisions and installed Analysis Artifacts.


## Before you run it

Install the repository with its development dependencies and generate the Th17 Compass example files as described in `demo/README.md`.


In [ ]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (  # noqa: E402
    AnalysisWorkspace,
    ExpressionData,
    ExpressionPreprocessingConfig,
    MetabolicModel,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
)
from metabis.version import __version__ as metabis_version  # noqa: E402


In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


def resolve_example_data_dir(repo_root: Path) -> Path:
    data_dir = repo_root / "data"

    if (
        (data_dir / "Th17_Compass_reactions.tsv").exists()
        and (data_dir / "Th17_Compass_expression.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
        and (data_dir / "Th17_Compass_metadata.tsv").exists()
    ):
        return data_dir

    raise FileNotFoundError(
        "Could not find the generated Th17 Compass example files in metabis/data."
    )


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = resolve_example_data_dir(repo_root)
reaction_data_path = data_dir / "Th17_Compass_reactions.tsv"
expression_data_path = data_dir / "Th17_Compass_expression.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"
metadata_path = data_dir / "Th17_Compass_metadata.tsv"

print(f"Metabis version: {metabis_version}")
print(f"Repository root: {repo_root}")
print(f"Data directory: {data_dir}")


## 1. Initialize the metabolic model

`MetabolicModel` exposes the packaged Human1-based reference used throughout the library. It is the object that knows about reactions, genes, subsystems, compartments, and identifier translation.

For most notebook workflows, the main things you do first are:

- build the model with caching enabled
- apply any identifier alignment corrections needed for your input data
- inspect the reaction, gene, and subsystem spaces that later objects will use

In [ ]:
model = MetabolicModel(use_cache=True)

alignment_file_path = model.get_alignment_file_path("RECON2")
alignment_results = model.apply_alignment_corrections(alignment_file_path)

print(
    f"Applied {alignment_results['corrections_applied']} reaction ID corrections from {Path(alignment_file_path).name}."
)
print(f"Model reactions: {len(model.get_reactions())}")
print(f"Model genes: {len(model.get_genes())}")
print(f"Model subsystems: {len(model.get_subsystems())}")

In [ ]:
example_subsystem = model.get_subsystems()[0]

print("Example reactions:", model.get_reactions()[:5])
print("Example genes:", model.get_genes()[:5])
print("Example subsystem:", example_subsystem)
print(
    "Example subsystem size:",
    len(model.get_reactions_for_subsystem(example_subsystem)),
)
print(
    "SAT1 translated to model gene ID:",
    model.translate_gene(
        "SAT1",
        input_format="geneSymbols",
        output_format="ensmbl_id",
    ),
)

## 2. Load the reaction matrix

`ReactionData` loads a reaction table, translates its identifiers into the model reaction space, and records preprocessing provenance.

The generated Compass file is tab-delimited, stores samples in columns, and uses separate `_pos` and `_neg` penalties. `ReactionPreprocessingConfig(compass_activity=True)` makes the first pipeline stage explicit: decode those penalties into non-negative activity scores. No optional scale transform is applied here.


In [ ]:
reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
    "preprocessing": ReactionPreprocessingConfig(compass_activity=True),
}

expression_loading_options = {
    "gene_format": "geneSymbols",
    "preprocessing": ExpressionPreprocessingConfig(normalize_total=True),
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}

reaction_loading_options, expression_loading_options


In [ ]:
reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)

reaction_df = reaction_data.get_reaction_df()
reaction_alignment = reaction_validation.get("alignment_report", {})

print(
    f"Reaction matrix: {reaction_df.shape[0]} samples x {reaction_df.shape[1]} reaction features"
)
print(f"Mapped reactions: {reaction_alignment.get('mapped_count', 0)}")
print(f"Unmapped reactions: {reaction_alignment.get('unmapped_count', 0)}")
print(f"Model coverage: {reaction_alignment.get('model_coverage_percent', 0):.2f}%")
reaction_validation

In [ ]:
print("First sample IDs:", reaction_data.get_sample_ids()[:5])
reaction_df.iloc[:3, :5]

## 3. Load the expression matrix

`ExpressionData` converts an expression table into the model gene space. In this example, the input file uses gene symbols, while the model stores Ensembl-like identifiers internally.

This class also handles normalization options, so it is the right place to keep basic preprocessing tied to the expression matrix itself.

In [ ]:
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

expression_df = expression_data.get_expression_df()
expression_alignment = expression_validation.get("alignment_report", {})

print(
    f"Expression matrix: {expression_df.shape[0]} samples x {expression_df.shape[1]} genes"
)
print(f"Mapped genes: {expression_alignment.get('mapped_count', 0)}")
print(f"Unmapped genes: {expression_alignment.get('unmapped_count', 0)}")
print(f"Model coverage: {expression_alignment.get('model_coverage_percent', 0):.2f}%")
expression_validation

In [ ]:
print("First sample IDs in expression data:", expression_df.index[:5].tolist())
expression_df.iloc[:3, :5]

## 4. Create the Analysis Workspace

The model, aligned matrices, and standalone sample annotations form the Metabolic Dataset. `AnalysisWorkspace` owns their revision identity and is the installation boundary for derived Analysis Artifacts.


In [ ]:
metadata = SampleMetadata.from_table(
    metadata_path,
    sample_id_column="sample",
)

dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
metadata_alignment = dm.update_sample_metadata(metadata)

shared_samples = sorted(
    set(reaction_data.get_sample_ids()) & set(expression_data.get_sample_ids())
)

print(f"Shared sample IDs across matrices: {len(shared_samples)}")
print(f"Metadata columns: {list(metadata.columns)}")
print(f"Strict metadata alignment: {metadata_alignment.is_strict_match}")
print(f"Current mode: {dm.get_mode()}")


## Takeaways

The Metabolic Dataset now has a model, reaction activities, expression values, and aligned sample annotations. The `AnalysisWorkspace` records that input state and is ready to define Sample Groups.

The next notebook creates metadata-driven Sample Groups and installs dimensionality-reduction Analysis Artifacts.
